# File Finder
Notebook with a helper that reads `boilest.db` and prints video file paths for each directory listed in the `directories` table.

In [1]:
import sqlite3
import os
from pathlib import Path

def scan_directories_from_db(db_path='boilest.db', extensions=None):
    """Read `directories` table from `db_path` and yield video file paths as found.
    Args:
        db_path (str|Path): path to sqlite database file.
        extensions (iterable): file extensions to consider as video files.
    """
    if extensions is None:
        extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']

    db_path = Path(db_path)
    if not db_path.exists():
        print(f'Database not found: {db_path}')
        return

    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()
    try:
        cur.execute("SELECT path FROM directories")
        rows = cur.fetchall()
    except Exception as e:
        print('Error reading directories table:', e)
        rows = []
    finally:
        conn.close()

    for (path,) in rows:
        path = os.path.expanduser(path)
        if not os.path.isdir(path):
            print(f'Directory not found: {path}')
            continue
        # Walk directory and yield matching files as they are found
        for root, dirs, files in os.walk(path):
            for file in files:
                for ext in extensions:
                    if file.lower().endswith(ext.lower()):
                        yield os.path.join(root, file)

# Example usage: iterate over generator:
# for fp in scan_directories_from_db('boilest.db'):
#     print(fp)

In [2]:
# Invoke the scanner using the default database path
# Print each result as it is yielded
for fp in scan_directories_from_db():
    print(fp)

/media\Media 1\test_file_01.mp4
/media\Media 1\Media A\test_file_02.mp4
/media\Media 1\Media A\test_file_03.mp4
/media\Media 2\test_file_04.mp4
/media\Media 2\test_file_05.mp4
/media\Media 3\test_file_06.mp4
/media\Media 4\test.mkv
/media\Media 4\test_file_07.MP4


# Probing
Probe using ffprobe for codec details

In [3]:
import subprocess
import json
import shutil
from pathlib import Path

def probe_file(filepath, timeout=30):
    """Run ffprobe on `filepath`, print and return a JSON summary of format and streams.
    Returns a dict with 'format' and 'streams' keys, or None on error.
    """
    filepath = Path(filepath)
    if not filepath.exists():
        print(f'File not found: {filepath}')
        return None
    if not shutil.which('ffprobe'):
        print('ffprobe not found in PATH')
        return None
    cmd = ["ffprobe","-v","quiet","-print_format","json","-show_format","-show_streams", str(filepath)]
    try:
        cp = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=True)
        data = json.loads(cp.stdout)
    except subprocess.CalledProcessError as e:
        print('ffprobe failed:', e)
        return None
    except Exception as e:
        print('Error running ffprobe:', e)
        return None

    summary = {}
    summary['format'] = data.get('format', {})
    streams = []
    for s in data.get('streams', []):
        streams.append({
            'index': s.get('index'),
            'codec_type': s.get('codec_type'),
            'codec_name': s.get('codec_name'),
            'codec_long_name': s.get('codec_long_name'),
            'width': s.get('width'),
            'height': s.get('height'),
            'sample_rate': s.get('sample_rate'),
            'channels': s.get('channels'),
            'bit_rate': s.get('bit_rate'),
            'duration': s.get('duration'),
            'tags': s.get('tags', {})
        })
    summary['streams'] = streams
    # Print JSON summary
    print(json.dumps(summary, indent=2))
    return summary

# Example:
# probe_file('path/to/video.mp4')

In [6]:
probe_file('/media/Media 1/test_file_01.mp4')

ffprobe not found in PATH
